# 🛸 Video Colorizer (Lost in Space) - Google Colab (GPU NVIDIA)

Cuaderno oficial para ejecutar el pipeline de colorización industrial de *Perdidos en el Espacio* en la nube con aceleración por GPU CUDA (NVIDIA T4, L4, V100 o A100).

Repositorio: [github.com/jdmarinv/video-colorizer](https://github.com/jdmarinv/video-colorizer)

### Características principales:
- **Preservación Total de Luminancia ($L$ en CIE LAB):** Cero pérdida de grano fílmico o microcontraste.
- **Propagación Temporal con Flujo Óptico:** Erradicación del parpadeo (*color boiling*).
- **Paleta Canónica:** Reglas derivadas de las temporadas en color (Temporada 2 y 3).
- **Integración con Google Drive:** Entrada y salida directa de video sin agotar almacenamiento local.

## Paso 1: Verificar Aceleración por GPU (NVIDIA CUDA)
Asegúrate de tener seleccionado un entorno con GPU en el menú:  
`Entorno de ejecución` > `Cambiar tipo de entorno de ejecución` > **T4 GPU** (o superior).

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU activa: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ ADVERTENCIA: No hay GPU activa. Ve a 'Entorno de ejecución' > 'Cambiar tipo de entorno' y selecciona GPU.")

## Paso 2: Montar Google Drive
Conecta tu cuenta de Google Drive para leer los archivos `.mkv` de entrada y guardar los episodios colorizados resultantes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Carpetas sugeridas en Google Drive
drive_base = Path('/content/drive/MyDrive/LostInSpace')
input_dir = drive_base / 'Input'
output_dir = drive_base / 'Colorized'

input_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

print(f"✅ Carpeta de entrada en Drive: {input_dir}")
print(f"✅ Carpeta de salida en Drive:  {output_dir}")
print("💡 Coloca aquí tus episodios en blanco y negro (.mkv o .mp4).")

## Paso 3: Configurar el Repositorio y Dependencias en Colab
Clona el código del proyecto desde `github.com/jdmarinv/video-colorizer` (o úsalo desde Drive si ya lo subiste allí).

In [ ]:
#@title ⚙️ Inicializar Código del Colorizador { display-mode: "form" }
source_mode = "Clonar desde GitHub / Copiar repositorio" #@param ["Clonar desde GitHub / Copiar repositorio", "Usar carpeta existente en Google Drive"]
github_repo_url = "https://github.com/jdmarinv/video-colorizer.git" #@param {type:"string"}
github_token = "" #@param {type:"string"} {help: "Opcional: Si el repositorio es privado, coloca aquí un Personal Access Token (PAT)"}
drive_project_path = "/content/drive/MyDrive/video-colorizer" #@param {type:"string"}

import os
import subprocess
from pathlib import Path

work_dir = Path("/content/video-colorizer")

if source_mode == "Usar carpeta existente en Google Drive":
    if Path(drive_project_path).exists():
        work_dir = Path(drive_project_path)
        print(f"Usando proyecto directamente desde Drive: {work_dir}")
    else:
        print(f"⚠️ No se encontró {drive_project_path}, procediendo a clonar...")
        clone_url = github_repo_url
        if github_token.strip():
            clone_url = github_repo_url.replace("https://", f"https://{github_token.strip()}@")
        !git clone {clone_url} /content/video-colorizer
        work_dir = Path("/content/video-colorizer")
else:
    if not work_dir.exists():
        clone_url = github_repo_url
        if github_token.strip():
            clone_url = github_repo_url.replace("https://", f"https://{github_token.strip()}@")
        !git clone {clone_url} /content/video-colorizer
    else:
        print("El repositorio ya existe en /content/video-colorizer. Actualizando con git pull...")
        !git -C /content/video-colorizer pull

%cd {work_dir}

# Instalar dependencias de Python
!pip install -q opencv-python tqdm

# Crear carpetas locales de trabajo
!mkdir -p models live_previews

# Descargar pesos del modelo grande (912 MB) si no existen
model_large = Path("models/ddcolor_modelscope.pth")
if not model_large.exists():
    print("Descargando modelo neuronal DDColor (912 MB)... Esto toma ~30 segundos en Colab.")
    !curl -L https://huggingface.co/piddnad/DDColor-models/resolve/main/ddcolor_modelscope.pth -o models/ddcolor_modelscope.pth
else:
    print("✅ Modelo DDColor ya descargado.")

print("\n🎉 Entorno configurado y listo para procesar.")

## Paso 4: Ejecutar la Colorización con Interfaz Gráfica (Formulario)
Ajusta los parámetros deseados y haz clic en reproducir (Play) para iniciar el procesamiento con aceleración GPU.

In [ ]:
#@title 🚀 Lanzar Colorizador de Episodios { display-mode: "form" }
target = "1" #@param {type:"string"} {help: "Número de episodio (ej: '1'), rango (ej: '1-3'), o 'all'"}
mode = "balanced" #@param ["balanced", "fast", "direct"]
sample_step = 8 #@param {type:"integer"}
crf = 18 #@param {type:"slider", min:14, max:28, step:1}
preset = "medium" #@param ["ultrafast", "fast", "medium", "slow"]
model_size = "large" #@param ["large", "tiny"]
chunk_size = 500 #@param {type:"integer"}
force = False #@param {type:"boolean"}
input_dir = "/content/drive/MyDrive/LostInSpace/Input" #@param {type:"string"}
output_dir = "/content/drive/MyDrive/LostInSpace/Colorized" #@param {type:"string"}

cmd = [
    "python", "colorize_episode.py", target,
    "--mode", mode,
    "--input-dir", input_dir,
    "--output-dir", output_dir,
    "--crf", str(crf),
    "--preset", preset,
    "--model-size", model_size,
    "--chunk-size", str(chunk_size)
]

if sample_step:
    cmd.extend(["--sample-step", str(sample_step)])
if force:
    cmd.append("--force")

print("Comando a ejecutar:", " ".join(cmd))
!{" ".join(cmd)}

## Paso 5: Previsualizar Resultados y Galería en Vivo
Puedes inspeccionar los fotogramas de previsualización que el script genera en tiempo real dentro de `live_previews/`.

In [ ]:
from pathlib import Path
from IPython.display import display, Image
import glob

preview_files = sorted(glob.glob("live_previews/*.jpg"))
if preview_files:
    print(f"Mostrando última previsualización en vivo: {preview_files[-1]}")
    display(Image(preview_files[-1]))
else:
    print("Aún no hay previsualizaciones en live_previews/.")